In [1]:
!pip -q install requests pandas tqdm

import os
import json
import random
import requests
import pandas as pd

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

In [20]:
OWNER = "pandas-dev"
REPO = "pandas"

ISSUE_LIMIT = 3000
GOLDEN_PER_LABEL = 25

OUTPUT_PATH = Path("/content/golden_dataset.json")

LABELS = ["bug", "feature", "docs", "question"]

GITHUB_TO_ASSIGNMENT_LABEL = {
    "bug": "bug",
    "enhancement": "feature",
    "docs": "docs",
    "usage question": "question",
}

In [21]:
def get_github_token():
    token = os.getenv("GITHUB_TOKEN")

    try:
        from google.colab import userdata
        colab_token = userdata.get("GITHUB_TOKEN")
        if colab_token:
            token = colab_token
    except Exception:
        pass

    return token

In [22]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

if token:
    print("GitHub token loaded successfully")
else:
    print("GitHub token NOT found")

GitHub token loaded successfully


fetch closed issues

In [23]:
def fetch_closed_issues(owner, repo, limit):
    token = get_github_token()

    headers = {
        "Accept": "application/vnd.github+json"
    }

    if token:
        headers["Authorization"] = f"Bearer {token}"

    rows = []
    page = 1

    while len(rows) < limit:
        url = f"https://api.github.com/repos/{owner}/{repo}/issues"

        params = {
            "state": "closed",
            "per_page": 100,
            "page": page,
            "sort": "updated",
            "direction": "desc",
            # Keeps the query smaller and avoids GitHub 422 pagination issues
            "since": "2020-01-01T00:00:00Z",
        }

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30,
        )

        if response.status_code == 422:
            print("Stopped because GitHub rejected further pagination.")
            print("Current page:", page)
            print("Rows collected:", len(rows))
            break

        response.raise_for_status()
        items = response.json()

        if not items:
            break

        for issue in items:
            if "pull_request" in issue:
                continue

            rows.append({
                "id": issue["id"],
                "number": issue["number"],
                "title": issue.get("title") or "",
                "body": issue.get("body") or "",
                "labels": [label.get("name", "") for label in issue.get("labels", [])],
                "state": issue.get("state"),
                "created_at": issue.get("created_at"),
                "closed_at": issue.get("closed_at"),
                "html_url": issue.get("html_url"),
                "comments": issue.get("comments", 0),
            })

            if len(rows) >= limit:
                break

        page += 1

    return rows


raw_issues = fetch_closed_issues(OWNER, REPO, ISSUE_LIMIT)
print(f"Fetched {len(raw_issues)} closed non-PR issues")

Fetched 3000 closed non-PR issues


map labels

In [24]:
def map_issue_labels(github_labels):
    mapped = set()

    for label in github_labels:
        normalized = label.lower().strip()

        if normalized in GITHUB_TO_ASSIGNMENT_LABEL:
            mapped.add(GITHUB_TO_ASSIGNMENT_LABEL[normalized])

    return [label for label in LABELS if label in mapped]


candidates = []

for issue in raw_issues:
    mapped = map_issue_labels(issue["labels"])

    #keep only issues with exactly one assignment label
    if len(mapped) != 1:
        continue

    label = mapped[0]
    text = f"{issue['title']}\n\n{issue['body']}".strip()

    if len(text) < 50:
        continue

    candidates.append({
        "id": issue["id"],
        "number": issue["number"],
        "title": issue["title"],
        "body": issue["body"],
        "text": text,
        "label": label,
        "source_repo": f"{OWNER}/{REPO}",
        "source_labels": issue["labels"],
        "html_url": issue["html_url"],
        "created_at": issue["created_at"],
        "closed_at": issue["closed_at"],
    })

print(f"Candidate golden examples: {len(candidates)}")
print(Counter(row["label"] for row in candidates))

Candidate golden examples: 2363
Counter({'bug': 1316, 'feature': 544, 'docs': 440, 'question': 63})


In [25]:
random.seed(42)

golden_rows = []

for label in LABELS:
    label_rows = [row for row in candidates if row["label"] == label]

    sample_size = min(GOLDEN_PER_LABEL, len(label_rows))
    sampled = random.sample(label_rows, sample_size)

    golden_rows.extend(sampled)

random.shuffle(golden_rows)

print(f"Golden set size: {len(golden_rows)}")
print(Counter(row["label"] for row in golden_rows))

Golden set size: 100
Counter({'feature': 25, 'bug': 25, 'docs': 25, 'question': 25})


In [26]:
preview_df = pd.DataFrame(golden_rows)[
    ["number", "label", "title", "source_labels", "html_url"]
]

preview_df.head(20)

,number,label,title,source_labels,html_url
0,60998,feature,ENH: Add `to_numeric_br()` function to convert...,"[Enhancement, Dtype Conversions, Needs Triage]",https://github.com/pandas-dev/pandas/issues/60998
1,64941,bug,"BUG: `str.replace("""", """")` sometimes hangs up ...","[Bug, Strings, Arrow]",https://github.com/pandas-dev/pandas/issues/64941
2,51027,docs,"BUG: converting to datetime with unit=D, unit=...",[Docs],https://github.com/pandas-dev/pandas/issues/51027
3,60438,question,QST: when I import pandas as pd then i get an ...,[Usage Question],https://github.com/pandas-dev/pandas/issues/60438
4,60083,docs,DOC: Negative values ​​of n in pandas.DataFram...,"[Docs, Closing Candidate]",https://github.com/pandas-dev/pandas/issues/60083
5,62653,bug,BUG: Series.str.replace stopped working with r...,"[Bug, Strings, Arrow]",https://github.com/pandas-dev/pandas/issues/62653
6,14601,docs,DOC: min_itemsize for HDFStore append for enc...,"[Docs, IO HDF5]",https://github.com/pandas-dev/pandas/issues/14601
7,10517,feature,Explore use of ReadStat for binary format read...,"[Enhancement, IO Data, IO SAS, IO Format Request]",https://github.com/pandas-dev/pandas/issues/10517
8,55435,docs,DOC: Add docstrings for MultiIndex.levels and ...,"[Docs, MultiIndex]",https://github.com/pandas-dev/pandas/issues/55435
9,41517,feature,QST: How to export json as zip without directo...,"[Enhancement, IO Data, Needs Discussion]",https://github.com/pandas-dev/pandas/issues/41517


In [27]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(golden_rows, f, indent=2, ensure_ascii=False)

print(f"Saved golden set to: {OUTPUT_PATH}")

Saved golden set to: /content/golden_dataset.json


In [28]:
from google.colab import files

files.download(str(OUTPUT_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>